# <font color="steelblue">24. La distribución normal</font>

*Unidad 4. Probabilidad* · Sesión teórica 24 de 30

**Adaptación para el Grado en Periodismo**: Fernando Borrás Rocher (f.borras@umh.es)

**Basado en SIMUSTAT**, de Fernando Borrás, Federico Botella, Inés Hernández, M.ª Asunción Martínez Mayoral, Josep Moltó y Javier Morales.
Departamento de Estadística, Matemáticas e Informática. Universidad Miguel Hernández de Elche.

**Asignatura**: Estadística (1200), 1.º curso, Grado en Periodismo.

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="CC BY-SA 4.0" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a> Creative Commons Atribución-CompartirIgual 4.0.

> ⚠️ **Los datos de este curso son simulados.** Imitan la forma de fuentes reales (INE, CIS, audímetros, agencias de verificación), pero ninguna cifra es real. Úsalos para aprender; para publicar, ve siempre a la fuente oficial.

## <font color='steelblue'>Antes de empezar: configura el cuaderno</font>

Ejecuta **una vez** la celda de abajo (botón ▶). Instala la librería de las preguntas, descarga los datos (`encuesta_medios.csv`) y las preguntas de autoevaluación. Si todo va bien verás **`Cuaderno configurado`**.

> 💡 Si Colab se desconecta, los ficheros se borran: vuelve a ejecutar esta celda.

In [ ]:
#@title <font color="steelblue" size="+1"> Ejecutar configuración { display-mode: "form" }
import os, sys, json, base64, subprocess, urllib.request

RAW = "https://raw.githubusercontent.com/fborrasumh/statperiodismo/main/"
DATOS = ["encuesta_medios.csv"]
PREGUNTAS = ["q_P24_1.json", "q_P24_2.json", "q_P24_3.json", "q_P24_4.json"]

try:
    from jupyterquiz import display_quiz
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupyterquiz==2.9.6.4"], check=True)
    from jupyterquiz import display_quiz

for carpeta, ficheros in (("datos/", DATOS), ("autoeval/", PREGUNTAS)):
    for f in ficheros:
        if not os.path.exists(f):
            urllib.request.urlretrieve(RAW + carpeta + f, f)

def pregunta(fichero):
    """Muestra una pregunta interactiva de autoevaluación."""
    display_quiz(json.loads(base64.b64decode(open(fichero, "rb").read()).decode("utf8")))

print("Cuaderno configurado")

## <font color='steelblue'>Introducción</font>

La campana de Gauss aparece en las alturas, en los errores de medida y, sobre todo, en **las encuestas**: si repitiéramos muchas veces una encuesta, sus resultados formarían una campana alrededor del valor real. Por eso la **distribución normal** es la base del margen de error. Este cuaderno es el puente entre la probabilidad y la inferencia.

**Qué necesitas saber antes**: los cuadernos 10 (desviación típica), 11 (puntuaciones z) y 23.

## <font color='steelblue'>Qué vas a aprender</font>

Al terminar este cuaderno sabrás:

1. Describir la distribución normal y sus parámetros.
2. Usar la regla 68–95–99,7.
3. Calcular probabilidades y percentiles con `scipy.stats.norm`.
4. Ver cómo la binomial se aproxima a la normal cuando n es grande.

## <font color='steelblue'>Contenidos</font>

1. La campana
2. La regla 68–95–99,7
3. Calcular probabilidades y percentiles
4. La binomial se vuelve normal

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 20)

## <font color='steelblue'>1. La campana</font>

Una variable **normal** $N(\mu, \sigma)$ es continua, simétrica y con forma de campana. Queda totalmente determinada por su **media** $\mu$ (el centro) y su **desviación típica** $\sigma$ (la anchura).

En una variable continua no tiene sentido la probabilidad de un valor exacto: se calcula la probabilidad de un **intervalo**, que es el **área bajo la curva**.

In [ ]:
x = np.linspace(-4, 4, 400)
for mu, sigma in [(0, 1), (0, 2), (1.5, 0.7)]:
    plt.plot(x, stats.norm(mu, sigma).pdf(x), label=f"N({mu}, {sigma})")
plt.legend()
plt.title("Tres normales: la media mueve, la desviación típica ensancha")
plt.show()

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P24_1.json")

## <font color='steelblue'>2. La regla 68–95–99,7</font>

En cualquier normal:

- el **68 %** de los valores está a menos de 1 σ de la media;
- el **95 %**, a menos de 2 σ (más exactamente 1,96 σ);
- el **99,7 %**, a menos de 3 σ.

In [ ]:
Z = stats.norm(0, 1)            # normal estándar
for k in [1, 1.96, 2, 3]:
    print(f"P(-{k} < Z < {k}) = {Z.cdf(k) - Z.cdf(-k):.4f}")

In [ ]:
xs = np.linspace(-4, 4, 400)
plt.plot(xs, Z.pdf(xs), color="black")
plt.fill_between(xs, Z.pdf(xs), where=np.abs(xs) <= 1.96, color="steelblue", alpha=0.4, label="95 % central")
plt.legend()
plt.title("El 95 % del área está entre −1,96 y 1,96")
plt.show()

Este **1,96** volverá a aparecer en el margen de error de las encuestas (cuaderno 27).

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P24_2.json")

## <font color='steelblue'>3. Calcular probabilidades y percentiles</font>

Para calcular con cualquier normal se **tipifica** (como en el cuaderno 11): $z = (x - \mu) / \sigma$. Con `scipy` no hace falta: `cdf(x)` da $P(X \le x)$ y `ppf(q)` da el valor que deja por debajo la proporción $q$.

In [ ]:
enc = pd.read_csv("encuesta_medios.csv")
conf = enc["confianza_medios"].dropna()
mu, sigma = conf.mean(), conf.std()
print(f"Confianza: media {mu:.2f}, desviación típica {sigma:.2f}")

modelo = stats.norm(mu, sigma)
print("P(confianza <= 3) según la normal:", round(modelo.cdf(3.5), 3))   # 3,5: corrección por ser valores enteros
print("Proporción real en los datos:      ", round((conf <= 3).mean(), 3))
print("Percentil 90 según la normal:", round(modelo.ppf(0.9), 1))

**Cómo leerlo**: la normal describe bastante bien la confianza en los medios (que es casi simétrica). En variables asimétricas, como los salarios, el modelo normal daría resultados malos: compruébalo siempre con un histograma.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P24_3.json")

## <font color='steelblue'>4. La binomial se vuelve normal</font>

Cuando $n$ es grande, la binomial $B(n, p)$ se parece mucho a una normal con la misma media ($np$) y la misma desviación típica ($\sqrt{np(1-p)}$). Es lo que pasa con el número de «síes» en una encuesta grande, y la razón de que el margen de error use la normal.

In [ ]:
n, p = 1000, 0.34
B = stats.binom(n, p)
N = stats.norm(n * p, np.sqrt(n * p * (1 - p)))
k = np.arange(280, 400)
plt.bar(k, B.pmf(k), color="lightgrey", label="Binomial B(1000; 0,34)")
plt.plot(k, N.pdf(k), color="firebrick", label="Normal aproximada")
plt.legend()
plt.xlabel("Personas que votarían a A entre 1.000 entrevistadas")
plt.show()

print(f"Entre el {(n*p - 1.96*np.sqrt(n*p*(1-p)))/n:.1%} y el {(n*p + 1.96*np.sqrt(n*p*(1-p)))/n:.1%} de los votos, con un 95 % de probabilidad")

**Qué ha pasado**: si el 34 % de la población votara a A, una encuesta de 1.000 personas daría un resultado entre el 31 % y el 37 % en el 95 % de las ocasiones. Esa horquilla de unos ±3 puntos es el **margen de error** del que hablaremos en la unidad 5.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P24_4.json")

## <font color='steelblue'>Reto aprendiz</font>

Resuelve estos ejercicios en la celda vacía de debajo de cada enunciado. Si te atascas, vuelve a la sección indicada. La solución está plegada: **inténtalo antes de mirarla**.

### **Ejercicio 1** *(sección 2)*

Si la altura de una población sigue una N(170, 8) cm, ¿qué porcentaje mide más de 186 cm? Hazlo primero con la regla 68–95–99,7 y luego con `scipy`.

<details><summary>Ver solución</summary>

```python
1 - stats.norm(170, 8).cdf(186)
```

186 es la media + 2σ: queda fuera alrededor de un 2,5 % (la mitad del 5 %).
</details>

### **Ejercicio 2** *(sección 3)*

Con el modelo normal de la confianza, ¿qué valor deja por encima al 10 % de las personas más confiadas?

<details><summary>Ver solución</summary>

```python
modelo.ppf(0.9)
```
</details>

### **Ejercicio 3** *(sección 3)*

Dibuja el histograma de la confianza (con `density=True`) y encima la curva normal ajustada.

<details><summary>Ver solución</summary>

```python
plt.hist(conf, bins=range(0, 12), density=True, align="left", color="lightgrey")
xs = np.linspace(0, 10, 200)
plt.plot(xs, modelo.pdf(xs), color="firebrick")
plt.show()
```
</details>

### **Ejercicio 4** *(sección 4)*

Repite la horquilla del 95 % para una encuesta de 400 personas y otra de 4.000 (con p = 0,34). ¿Qué le pasa al margen?

<details><summary>Ver solución</summary>

```python
for n_ in [400, 4000]:
    e = 1.96 * np.sqrt(p * (1 - p) / n_)
    print(n_, round(p - e, 3), round(p + e, 3))
```

Al multiplicar por 10 el tamaño, el margen se divide por √10 ≈ 3,2.
</details>

## <font color='steelblue'>Resumen: ideas clave</font>

- La **normal** $N(\mu, \sigma)$ es simétrica, con forma de campana; $\mu$ la centra y $\sigma$ la ensancha.
- Regla **68–95–99,7**: a 1, 2 y 3 desviaciones típicas de la media; el 95 % exacto está a ±1,96 σ.
- En `scipy.stats.norm`: `cdf` da probabilidades acumuladas y `ppf` percentiles.
- Con $n$ grande, la **binomial** se aproxima por una normal: es la base del **margen de error**.
- Antes de usar la normal como modelo, comprueba con un histograma que los datos son aproximadamente simétricos.

## <font color='steelblue'>Glosario</font>

| Término | Significado |
| :-- | :-- |
| **Distribución normal** | Distribución continua simétrica en forma de campana. |
| **Normal estándar** | Normal con media 0 y desviación típica 1. |
| **Tipificar** | Restar la media y dividir por la desviación típica. |
| **`ppf`** | Función que devuelve el valor de un percentil. |

## <font color='steelblue'>Errores frecuentes y cómo resolverlos</font>

| Síntoma | Causa probable | Solución |
| :-- | :-- | :-- |
| `NameError: name 'pd' is not defined` | No se ejecutó la celda de importación | Ejecuta las celdas desde el principio, en orden |
| `FileNotFoundError` al leer un fichero | No se ejecutó la configuración o Colab se reinició | Ejecuta la celda *Configura el cuaderno* |
| La pregunta de autoevaluación no aparece | Falta la configuración | Ejecuta la celda de configuración y luego la pregunta |

## <font color='steelblue'>Para seguir</font>

Con esto termina la unidad 4. En el cuaderno 25 empieza la inferencia: población, muestra y tipos de muestreo.

**Siguiente cuaderno**: [25. Población, muestra y tipos de muestreo](https://colab.research.google.com/github/fborrasumh/statperiodismo/blob/main/cuadernos/P25_poblacion_muestra_y_muestreo.ipynb)